1. На основе учебного ноутбука проведите финальную подготовку данных. Иизмените количество сегментирующих классов с `16` на `5`.

2. Проведите суммарно не менее `10` экспериментов и визуализируйте их результаты (включая точность обучения сетей на одинаковом количестве эпох, например, на `7`):

  - изменив `filters` в сверточных слоях
  - изменив `kernel_size` в сверточных слоях
  - изменив активационную функцию в скрытых слоях с `relu` на `linear` или/и `selu`, `elu`.

In [ ]:
import os
import gc
import time
import zipfile
import gdown
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
dataset_url = "https://storage.yandexcloud.net/aiueducation/Content/base/l14/construction_256x192.zip"
archive_name = "construction_256x192.zip"

if not os.path.exists(archive_name):
    gdown.download(dataset_url, archive_name, quiet=False)

with zipfile.ZipFile(archive_name, "r") as archive:
    archive.extractall(".")

print("Датасет загружен и распакован")

In [ ]:
IMG_WIDTH = 192
IMG_HEIGHT = 256

TRAIN_DIR = "train"
VAL_DIR = "val"

FLOOR = (100, 100, 100)
CEILING = (0, 0, 100)
WALL = (0, 100, 0)
COLUMN = (100, 0, 0)
APERTURE = (0, 100, 100)
DOOR = (100, 0, 100)
WINDOW = (100, 100, 0)
EXTERNAL = (200, 200, 200)
RAILINGS = (0, 200, 0)
BATTERY = (200, 0, 0)
PEOPLE = (0, 200, 200)
LADDER = (0, 0, 200)
INVENTORY = (200, 0, 200)
LAMP = (200, 200, 0)
WIRE = (0, 100, 200)
BEAM = (100, 0, 200)

CLASS_GROUPS = {
    0: [WALL, COLUMN, BEAM],
    1: [FLOOR, CEILING],
    2: [APERTURE, DOOR, WINDOW],
    3: [RAILINGS, BATTERY, LADDER, INVENTORY, LAMP, WIRE],
    4: [EXTERNAL, PEOPLE],
}

def read_image_list(root_dir, subfolder):
    folder = os.path.join(root_dir, subfolder)
    names = sorted(os.listdir(folder))
    result = []

    for filename in names:
        full_path = os.path.join(folder, filename)
        img = Image.open(full_path).convert("RGB").resize((IMG_HEIGHT, IMG_WIDTH))
        result.append(img)

    return result

def make_image_tensor(images):
    values = []

    for img in images:
        arr = np.asarray(img, dtype=np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        values.append(arr)

    return torch.tensor(np.stack(values), dtype=torch.float32)

def make_mask_tensor(masks):
    prepared = []

    for mask in masks:
        rgb = np.asarray(mask.convert("RGB").resize((IMG_HEIGHT, IMG_WIDTH)))
        class_map = np.zeros((IMG_WIDTH, IMG_HEIGHT), dtype=np.int64)

        for class_index, colors in CLASS_GROUPS.items():
            for color in colors:
                same_color = np.all(rgb == color, axis=-1)
                class_map[same_color] = class_index

        prepared.append(class_map)

    return torch.tensor(np.stack(prepared), dtype=torch.long)

train_source_images = read_image_list(TRAIN_DIR, "original")
val_source_images = read_image_list(VAL_DIR, "original")

x_train = make_image_tensor(train_source_images)
x_val = make_image_tensor(val_source_images)

del train_source_images, val_source_images
gc.collect()

train_source_masks = read_image_list(TRAIN_DIR, "segment")
val_source_masks = read_image_list(VAL_DIR, "segment")

y_train = make_mask_tensor(train_source_masks)
y_val = make_mask_tensor(val_source_masks)

del train_source_masks, val_source_masks
gc.collect()

print("x_train:", tuple(x_train.shape))
print("y_train:", tuple(y_train.shape))
print("x_val:", tuple(x_val.shape))
print("y_val:", tuple(y_val.shape))

In [ ]:
class SmallSegNet(nn.Module):
    def __init__(self, base_filters, kernel_size, activation_name):
        super().__init__()

        padding = kernel_size // 2
        self.activation = self._choose_activation(activation_name)

        self.encoder_conv = nn.Conv2d(3, base_filters, kernel_size=kernel_size, padding=padding)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.center_conv = nn.Conv2d(base_filters, base_filters * 2, kernel_size=kernel_size, padding=padding)

        self.up = nn.ConvTranspose2d(base_filters * 2, base_filters, kernel_size=2, stride=2)

        self.decoder_conv = nn.Conv2d(base_filters * 2, base_filters, kernel_size=kernel_size, padding=padding)
        self.output_conv = nn.Conv2d(base_filters, 5, kernel_size=1)

    def _choose_activation(self, name):
        variants = {
            "relu": nn.ReLU(),
            "linear": nn.Identity(),
            "selu": nn.SELU(),
            "elu": nn.ELU()
        }
        return variants[name]

    def forward(self, x):
        enc = self.activation(self.encoder_conv(x))
        middle = self.activation(self.center_conv(self.pool(enc)))
        restored = self.up(middle)
        joined = torch.cat([enc, restored], dim=1)
        decoded = self.activation(self.decoder_conv(joined))
        return self.output_conv(decoded)

In [ ]:
def pixel_accuracy(logits, target):
    prediction = torch.argmax(logits, dim=1)
    correct = (prediction == target).sum().item()
    total = target.numel()
    return correct / total

def run_training(model, train_loader, val_loader, epochs_count, learning_rate, device):
    model.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_accuracy_log = []
    val_accuracy_log = []

    for epoch in range(1, epochs_count + 1):
        model.train()
        train_acc_sum = 0
        train_batches = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = loss_function(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_acc_sum += pixel_accuracy(outputs.detach(), batch_y)
            train_batches += 1

        model.eval()
        val_acc_sum = 0
        val_batches = 0

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)

                outputs = model(batch_x)
                val_acc_sum += pixel_accuracy(outputs, batch_y)
                val_batches += 1

        train_acc = train_acc_sum / train_batches
        val_acc = val_acc_sum / val_batches

        train_accuracy_log.append(train_acc)
        val_accuracy_log.append(val_acc)

        print(f"Эпоха {epoch}/{epochs_count} | accuracy: {train_acc:.4f} | val_accuracy: {val_acc:.4f}")

    return {
        "accuracy": train_accuracy_log,
        "val_accuracy": val_accuracy_log
    }

def reset_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_experiment_title(index, config):
    return f"Опыт {index}: F={config['filters']}, K={config['kernel']}, A={config['activation']}"

In [ ]:
experiment_plan = [
    {"filters": 16, "kernel": 3, "activation": "relu"},
    {"filters": 32, "kernel": 3, "activation": "relu"},
    {"filters": 64, "kernel": 3, "activation": "relu"},
    {"filters": 32, "kernel": 5, "activation": "relu"},
    {"filters": 32, "kernel": 7, "activation": "relu"},
    {"filters": 32, "kernel": 3, "activation": "linear"},
    {"filters": 32, "kernel": 3, "activation": "selu"},
    {"filters": 32, "kernel": 3, "activation": "elu"},
    {"filters": 16, "kernel": 5, "activation": "elu"},
    {"filters": 64, "kernel": 5, "activation": "selu"},
]

EPOCHS_COUNT = 7
BATCH_SIZE = 8
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

history_logs = {}
final_accuracies = {}

for exp_number, params in enumerate(experiment_plan, start=1):
    print(f"\nЗапуск эксперимента {exp_number} из {len(experiment_plan)}")
    print(f"Параметры: filters={params['filters']}, kernel={params['kernel']}, activation={params['activation']}")

    reset_memory()
    time.sleep(1)

    model = SmallSegNet(
        base_filters=params["filters"],
        kernel_size=params["kernel"],
        activation_name=params["activation"]
    )

    history = run_training(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs_count=EPOCHS_COUNT,
        learning_rate=LEARNING_RATE,
        device=device
    )

    experiment_name = get_experiment_title(exp_number, params)
    val_accuracy_values = history["val_accuracy"]

    history_logs[experiment_name] = val_accuracy_values
    final_accuracies[experiment_name] = val_accuracy_values[-1]

    print(f"Эксперимент завершён. Итоговая val_accuracy: {final_accuracies[experiment_name]:.4f}")

    del model, history
    reset_memory()

print("\nВсе 10 экспериментов завершены.")

In [ ]:
def show_accuracy_comparison(history_dict, epochs_count):
    plt.figure(figsize=(15, 8))

    epochs = range(1, epochs_count + 1)

    for experiment_name, values in history_dict.items():
        plt.plot(epochs, values, marker="o", linewidth=2, label=experiment_name)

    plt.title("Сравнение val_accuracy для 10 экспериментов", fontsize=16, fontweight="bold")
    plt.xlabel("Эпоха обучения", fontsize=12)
    plt.ylabel("Точность на проверочной выборке", fontsize=12)
    plt.xticks(list(epochs))
    plt.grid(True)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
    plt.tight_layout()
    plt.show()

def print_model_rating(scores):
    print("\nРейтинг моделей по итоговой val_accuracy:")

    sorted_scores = sorted(scores.items(), key=lambda item: item[1], reverse=True)

    for place, (model_name, score) in enumerate(sorted_scores, start=1):
        print(f"{place}. {model_name} -> {score:.4f}")

show_accuracy_comparison(history_logs, EPOCHS_COUNT)
print_model_rating(final_accuracies)